Upload file train và test tại đây

In [ ]:
from google.colab import files
files.upload()

Saving test_dataset_1e-8.txt to test_dataset_1e-8.txt
Saving train_val_dataset_1e-8.txt to train_val_dataset_1e-8.txt


{'test_dataset_1e-8.txt': b'RID rs11585858_A rs4844600_A rs12037841_T rs4266886_T rs4562624_A rs6656401_A rs6661489_T rs1752684_A rs679515_T rs3818361_A rs6701713_A rs2093761_A rs2093760_A rs11576522_A rs12036785_C rs2296160_A rs61822977_G rs10863417_A rs10863418_C rs10779335_C rs10863420_A rs1830763_C rs1408078_T rs4844610_A rs1408077_A rs6697005_G rs11118328_C rs1060743_G rs10194375_A rs72846701_A rs6754293_T rs10200967_C rs17014923_T rs58402148_T rs11694743_G rs10929006_A rs4663095_A rs35114168_A rs11887884_T rs1530047_C rs76168490_G rs10207708_A rs11694264_A rs56117224_A rs56405130_A rs10166461_A rs749007_T rs749006_A rs55654668_T rs11682186_T rs72838215_A rs6431219_T rs744368_T rs11554586_A rs13032148_A rs6743470_T rs13410629_T rs13400939_A rs55646068_G rs10182292_A rs4663098_T rs7594230_A rs3845674_T rs10929007_A rs4663099_C rs4663100_C rs10929009_T rs11904144_G rs6431220_A rs7566991_A rs7601287_A rs7564197_A rs35103166_C rs72838226_C rs55669136_G rs7559175_G rs57724183_C rs75752

In [ ]:

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    BaggingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay,
)

# Config
TRAIN_FILE  = "train_val_dataset_1e-8.txt"   # train file
TEST_FILE   = "test_dataset_1e-8.txt"         # test file
OUTPUT_ROOT = "AutoEncoder"                   # output root file
THRESHOLD   = 0.5

LATENT_DIM  = 64       # Bottleneck Dimension
BATCH_SIZE  = 64
MAX_EPOCHS  = 200
LR_AE       = 1e-3     # learning rate AutoEncoder
WEIGHT_DECAY= 1e-4
PATIENCE    = 50        # early stopping patience

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device : {DEVICE}")
print(f"[INFO] Output : {OUTPUT_ROOT}/")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 – DATA PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

train_df = pd.read_csv(TRAIN_FILE, sep=" ")
test_df  = pd.read_csv(TEST_FILE,  sep=" ")
print(f"  Train shape : {train_df.shape}")
print(f"  Test  shape : {test_df.shape}")

X_train_raw = train_df.drop(columns=["RID", "Phenotype"]).values.astype(np.float32)
y_train     = train_df["Phenotype"].values.astype(int)
X_test_raw  = test_df.drop(columns=["RID",  "Phenotype"]).values.astype(np.float32)
y_test      = test_df["Phenotype"].values.astype(int)

INPUT_DIM = X_train_raw.shape[1]
print(f"  SNP features: {INPUT_DIM}")
print(f"  Train labels – Control:{(y_train==0).sum()}  Alzheimer:{(y_train==1).sum()}")
print(f"  Test  labels – Control:{(y_test ==0).sum()}  Alzheimer:{(y_test ==1).sum()}")

scaler     = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test_sc  = scaler.transform(X_test_raw).astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 – AUTOENCODER (PyTorch)
# ══════════════════════════════════════════════════════════════════════════════


class LightweightAutoEncoder(nn.Module):
    """
    Encoder: INPUT → Linear(256) + ReLU + Dropout(0.3)
                   → Linear(128) + ReLU
                   → Linear(64)           ← linear latent space (no activation)
    Decoder: Linear(128) + ReLU
           → Linear(256) + ReLU
           → Linear(INPUT) + Sigmoid
    """
    def __init__(self, input_dim: int, latent_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


# DataLoader – toàn bộ dữ liệu scaled (unsupervised)
ae_dataset  = TensorDataset(torch.tensor(X_train_sc))
n_val       = max(1, int(0.1 * len(ae_dataset)))
n_train_ae  = len(ae_dataset) - n_val
ae_train_ds, ae_val_ds = random_split(
    ae_dataset, [n_train_ae, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(ae_train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ae_val_ds,   batch_size=BATCH_SIZE, shuffle=False)

model     = LightweightAutoEncoder(INPUT_DIM, LATENT_DIM).to(DEVICE)
criterion = nn.MSELoss()
#criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_AE, weight_decay=WEIGHT_DECAY)

print(f"  Parameters   : {sum(p.numel() for p in model.parameters()):,}")
print(f"  Epochs={MAX_EPOCHS} | Batch={BATCH_SIZE} | LR={LR_AE} | Patience={PATIENCE}\n")

best_val_loss    = float("inf")
patience_counter = 0
best_state_dict  = None
train_losses, val_losses = [], []

for epoch in range(1, MAX_EPOCHS + 1):
    # Train
    model.train()
    epoch_loss = 0.0
    for (xb,) in train_loader:
        xb = xb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), xb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(ae_train_ds)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.to(DEVICE)
            val_loss += criterion(model(xb), xb).item() * xb.size(0)
    val_loss /= len(ae_val_ds)

    train_losses.append(epoch_loss)
    val_losses.append(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{MAX_EPOCHS} | Train Loss: {epoch_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Early Stopping
    if val_loss < best_val_loss - 1e-6:
        best_val_loss    = val_loss
        patience_counter = 0
        best_state_dict  = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n  ⏹  Early stopping tại epoch {epoch}  (best val loss = {best_val_loss:.6f})")
            break

model.load_state_dict(best_state_dict)
print("\n  AutoEncoder training complete  ✓")

# Lưu AE loss curve
os.makedirs(OUTPUT_ROOT, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss", color="#2196F3", lw=2)
ax.plot(val_losses,   label="Val Loss",   color="#FF9800", lw=2, linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.set_title("AutoEncoder – Training & Validation Loss")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_ROOT, "ae_loss_curve.pdf"), dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {OUTPUT_ROOT}/ae_loss_curve.pdf")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 – FEATURE EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

for param in model.parameters():
    param.requires_grad = False
model.eval()

def extract_features(X_np: np.ndarray) -> np.ndarray:
    t = torch.tensor(X_np, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        z = model.encoder(t)
    return z.cpu().numpy()

Z_train = extract_features(X_train_sc)
Z_test  = extract_features(X_test_sc)
print(f"  Encoded train : {Z_train.shape}")
print(f"  Encoded test  : {Z_test.shape}")
print("  Feature extraction complete  ✓")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 – CLASSIFICATION (Scikit-learn)
# ══════════════════════════════════════════════════════════════════════════════

classifiers = {
    "LR"              : LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED),
    "SVM_RBF"         : SVC(kernel="rbf",    class_weight="balanced", probability=True, random_state=SEED),
    "SVM_Linear"      : SVC(kernel="linear", class_weight="balanced", probability=True, random_state=SEED),
    "RandomForest"    : RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                               max_features="sqrt", random_state=SEED, n_jobs=-1),
}

for name in classifiers:
    classifiers[name].fit(Z_train, y_train)
    print(f"  {name} trained  ✓")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 – EVALUATION & VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════


def evaluate_and_plot(name: str,
                      y_true: np.ndarray,
                      y_probs: np.ndarray,
                      threshold: float,
                      model_dir: str):

    os.makedirs(model_dir, exist_ok=True)
    y_preds = (y_probs >= threshold).astype(int)

    metrics = {
        "Accuracy" : accuracy_score(y_true, y_preds),
        "Precision": precision_score(y_true, y_preds, zero_division=0),
        "Recall"   : recall_score(y_true, y_preds,    zero_division=0),
        "F1-Score" : f1_score(y_true, y_preds,         zero_division=0),
        "AUC-ROC"  : roc_auc_score(y_true, y_probs),
    }

    # ── 1. Confusion Matrix ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_preds,
        display_labels=["Control", "Alzheimer"],
        cmap="Blues", values_format="d", ax=ax,
    )
    ax.set_title(f"Confusion Matrix – {name}", fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "confusion_matrix.pdf"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ── 2. ROC Curve ──────────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="darkorange", lw=2,
            label=f"ROC curve (AUC = {metrics['AUC-ROC']:.4f})")
    ax.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve – {name}", fontsize=12)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "roc_curve.pdf"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ── 3. Risk Score Distribution (KDE) ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.kdeplot(y_probs[y_true == 0], label="Control",
                fill=True, color="green", alpha=0.5, ax=ax)
    sns.kdeplot(y_probs[y_true == 1], label="Alzheimer",
                fill=True, color="red", alpha=0.5, ax=ax)
    ax.axvline(x=threshold, color="black", linestyle="--", lw=1.5,
               label=f"Threshold = {threshold}")
    ax.set_xlabel("Predicted Probability")
    ax.set_ylabel("Density")
    ax.set_title(f"Risk Score Distribution – {name}", fontsize=12)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(model_dir, "risk_score_distribution.pdf"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    return metrics, y_preds


# ── Vòng lặp đánh giá ─────────────────────────────────────────────────────────
all_metrics  = []
predictions_df = pd.DataFrame({"y_true": y_test})

print(f"\n  {'Model':<20} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}")
print("  " + "─"*55)

for name, clf in classifiers.items():
    y_prob = clf.predict_proba(Z_test)[:, 1]
    model_dir = os.path.join(OUTPUT_ROOT, name)

    metrics, y_preds = evaluate_and_plot(
        name, y_test, y_prob, THRESHOLD, model_dir
    )

    print(f"  {name:<20} {metrics['Accuracy']:>7.4f} {metrics['Precision']:>7.4f}"
          f" {metrics['Recall']:>7.4f} {metrics['F1-Score']:>7.4f} {metrics['AUC-ROC']:>7.4f}")

    all_metrics.append({
        "Model"    : name,
        "Threshold": THRESHOLD,
        **{k: round(v, 4) for k, v in metrics.items()},
    })

    predictions_df[f"{name}_prob"] = y_prob
    predictions_df[f"{name}_pred"] = y_preds

print("  " + "─"*55)

# ── Lưu CSV ───────────────────────────────────────────────────────────────────
metrics_df = pd.DataFrame(all_metrics).sort_values("AUC-ROC", ascending=False)
metrics_df.to_csv(os.path.join(OUTPUT_ROOT, "metrics_summary.csv"), index=False)
predictions_df.to_csv(os.path.join(OUTPUT_ROOT, "test_predictions.csv"), index=False)

print(f"\n  Saved: {OUTPUT_ROOT}/metrics_summary.csv")
print(f"  Saved: {OUTPUT_ROOT}/test_predictions.csv")

# ── In bảng kết quả cuối ──────────────────────────────────────────────────────
print("\n" + "="*70)
print("  FINAL RESULTS (sorted by AUC-ROC)")
print("="*70)
print(f"  {'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}")
print("  " + "─"*60)
for _, row in metrics_df.iterrows():
    star = "  ◀ BEST" if row["Model"] == metrics_df.iloc[0]["Model"] else ""
    print(f"  {row['Model']:<22} {row['Accuracy']:>7.4f} {row['Precision']:>7.4f}"
          f" {row['Recall']:>7.4f} {row['F1-Score']:>7.4f} {row['AUC-ROC']:>7.4f}{star}")
print("="*70)

# ── Tổng kết file output ──────────────────────────────────────────────────────
import glob
all_pdfs = sorted(glob.glob(os.path.join(OUTPUT_ROOT, "**", "*.pdf"), recursive=True))
print(f"\n  ✓ Tổng cộng {len(all_pdfs)} PDF files")
print(f"  ✓ 2 CSV files (metrics_summary, test_predictions)")
print(f"\n  Cấu trúc thư mục:")
print(f"  {OUTPUT_ROOT}/")
print(f"  ├── ae_loss_curve.pdf")
print(f"  ├── metrics_summary.csv")
print(f"  ├── test_predictions.csv")
for name in classifiers:
    print(f"  ├── {name}/")
    print(f"  │   ├── confusion_matrix.pdf")
    print(f"  │   ├── roc_curve.pdf")
    print(f"  │   └── risk_score_distribution.pdf")

print("\n  PIPELINE COMPLETE  ✓")

[INFO] Device : cuda
[INFO] Output : AutoEncoder/

STEP 1 – DATA PREPROCESSING
  Train shape : (1000, 1726)
  Test  shape : (174, 1726)
  SNP features: 1724
  Train labels – Control:486  Alzheimer:514
  Test  labels – Control:84  Alzheimer:90
  MinMaxScaler [0, 1] applied  ✓

STEP 2 – AUTOENCODER (PyTorch)
  Architecture : 1724 → 256 → 128 → 64 → 128 → 256 → 1724
  Parameters   : 967,164
  Epochs=200 | Batch=64 | LR=0.001 | Patience=50

  Epoch    1/200 | Train Loss: 0.128501 | Val Loss: 0.097396
  Epoch   10/200 | Train Loss: 0.066941 | Val Loss: 0.068926
  Epoch   20/200 | Train Loss: 0.052698 | Val Loss: 0.053891
  Epoch   30/200 | Train Loss: 0.044845 | Val Loss: 0.047790
  Epoch   40/200 | Train Loss: 0.042116 | Val Loss: 0.044440
  Epoch   50/200 | Train Loss: 0.038071 | Val Loss: 0.041589
  Epoch   60/200 | Train Loss: 0.036479 | Val Loss: 0.040815
  Epoch   70/200 | Train Loss: 0.034481 | Val Loss: 0.039835
  Epoch   80/200 | Train Loss: 0.032088 | Val Loss: 0.038541
  Epoch   

In [ ]:
import shutil
from google.colab import files

# Create a zip archive of the AutoEncoder directory
output_filename = 'AutoEncoder_results_1e-8'
shutil.make_archive(output_filename, 'zip', OUTPUT_ROOT)

print(f"Directory '{OUTPUT_ROOT}' compressed to '{output_filename}.zip'")

# Download the zip file
files.download(f'{output_filename}.zip')

Directory 'AutoEncoder' compressed to 'AutoEncoder_results_1e-8_200_50.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>